> 在线讲义：[第 1 章 · 机器人上的推理：和云上有什么不同](https://wikieden.github.io/robot-edge-inference/lessons/01-robot-constraints.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 1 章 · 机器人上的推理：和云上有什么不同

配套讲义：仓库里 `ai-infra-edge/notes/lessons/01-robot-constraints.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（π0 的数字来自论文 Table I，GR00T N1 的频率来自论文 §2）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 π0 耗时分解 | 三段相加 | `sum(PI0_PARTS_MS.values())` | 73 ms；板外 86 ms |
| 第二步 控制周期 | $1000 / 50$ | `period_ms(50)` | 20 ms |
| 第二步 推理间隔 | $25 \times 20$ | `infer_interval_ms(25, 50)` | 500 ms → 2 Hz |
| 第二步 推理占比 | $73 / 500$ | `busy_fraction(73, 500)` | 14.6% |
| 第二步 GR00T N1 System 1 | $1000 / 120$ | `period_ms(120)` | 8.33 ms |

π0 的耗时分解来自论文 Table I（RTX 4090），GR00T N1 的频率来自论文 §2，都不是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
PI0_PARTS_MS = {"图像编码": 14, "观测前向": 32, "10 步 flow 动作": 27}  # π0 Table I，RTX 4090
PI0_NETWORK_MS = 13  # 走 Wi-Fi 放到板外时多出的网络时间

**函数 `period_ms`**：频率换成周期：每秒 hz 次，每次隔 1000 / hz 毫秒。

In [ ]:
def period_ms(hz: float) -> float:
    """频率换成周期：每秒 hz 次，每次隔 1000 / hz 毫秒。"""
    return 1000.0 / hz

**函数 `infer_interval_ms`**：每执行 steps_between 步动作推理一次，两次推理之间隔多久。

In [ ]:
def infer_interval_ms(steps_between: int, control_hz: float) -> float:
    """每执行 steps_between 步动作推理一次，两次推理之间隔多久。"""
    return steps_between * period_ms(control_hz)

**函数 `busy_fraction`**：推理时间占两次推理间隔的几成。

In [ ]:
def busy_fraction(infer_ms: float, interval_ms: float) -> float:
    """推理时间占两次推理间隔的几成。"""
    return infer_ms / interval_ms

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：π0 在 RTX 4090 上一次推理的耗时分解（论文 Table I）")
total = sum(PI0_PARTS_MS.values())
for k, v in PI0_PARTS_MS.items():
    print(f"  {k:<10} {v:>3} ms")
print(f"  板上合计 {total} ms；放到板外走 Wi-Fi 再加 {PI0_NETWORK_MS} ms → {total + PI0_NETWORK_MS} ms")

In [ ]:
print("\n第二步：控制 50 Hz、动作块 50 步、每执行 25 步推理一次")
print(f"  控制周期 {period_ms(50):.0f} ms")
gap = infer_interval_ms(25, 50)
print(f"  每 {gap:.0f} ms 推理一次 → 推理频率 {1000 / gap:.0f} Hz，控制频率仍是 50 Hz")
print(f"  推理 {total} ms 只占这段间隔的 {busy_fraction(total, gap):.1%}")

In [ ]:
print("\n第二步：GR00T N1 的两套频率（论文 §2）")
print(f"  System 2（看图想事）10 Hz → 每 {period_ms(10):.0f} ms 一次")
print(f"  System 1（出动作）120 Hz → 每 {period_ms(120):.2f} ms 一次")

## 3. 代码里才看得到的

- 单位统一用毫秒：频率进来先换成周期，后面全部是 ms 相加相除，不容易错。
- 论文 Table I 是在 RTX 4090 上测的，不是在机器人的机载板子上；搬到 Jetson 上会慢好几倍，第 3、8 章再算。

## 4. 自己改一改

- 把控制频率改成 100 Hz，看推理间隔和占比怎么变。
- 把 `PI0_PARTS_MS` 里 10 步 flow 的 27 ms 减半（假设只用 5 步），看板上合计变成多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 云上一锅炒得多，车上一份要准时；
> 动作成块换时间，块没用完就回来；
> 批量为一没人摊，带宽先算别忘掉；
> 电热网断都要想，出错先要能停好。